# Section 6 · AgentCore Runtime — From Local Agent to Deployed Endpoint (Groq edition)

> **Groq edition.** Same lifecycle as `deploy_to_agentcore_runtime.ipynb`; the agent's model runs on Groq instead of Bedrock. You still need AWS credentials (AgentCore Runtime, ECR, CodeBuild and IAM are AWS services), but no Bedrock model access. Set `GROQ_API_KEY` in your environment or a `.env` file.

The full lifecycle of deploying a Strands agent to Amazon Bedrock AgentCore Runtime:

1. Write the agent with `BedrockAgentCoreApp`
2. Test it locally
3. Configure the deployment
4. Launch to AgentCore (built in the cloud by CodeBuild — no local Docker needed)
5. Invoke it (starter toolkit and boto3)
6. Manage sessions
7. Clean up

**Prerequisites:** AWS credentials, a `GROQ_API_KEY`, and IAM permissions to create IAM roles, ECR repositories, CodeBuild projects, S3 buckets and AgentCore runtimes.

## Step 0: Install the deployment tooling (on your machine, not in the agent)

In [ ]:
%pip install -q bedrock-agentcore-starter-toolkit 'bedrock-agentcore>=1.24.0' 'strands-agents[openai]>=1.60.0' python-dotenv boto3

## Step 1: The agent code

`agent_groq.py` in this folder is a normal Strands agent plus four lines of AgentCore boilerplate. The cell below writes the same file, so you can edit it here and redeploy.

In [ ]:
%%writefile agent_groq.py
"""Document analysis agent for Amazon Bedrock AgentCore Runtime, powered by Groq.

Same agent as agent.py; only the model provider changes. Groq exposes an
OpenAI-compatible API, so Strands' OpenAIModel talks to it via `base_url`.
The deployed runtime needs GROQ_API_KEY in its environment (see the notebook).

Compared with a plain Strands agent, only four things are added:
  1. import BedrockAgentCoreApp
  2. app = BedrockAgentCoreApp()
  3. @app.entrypoint on the handler
  4. app.run() in __main__

Local test:
    python agent_groq.py      (needs GROQ_API_KEY, or a .env file)
    curl -X POST http://localhost:8080/invocations -H "Content-Type: application/json" \
         -d '{"prompt": "What are the key clauses in a standard NDA?"}'
"""

import os

from bedrock_agentcore import BedrockAgentCoreApp
from dotenv import load_dotenv
from strands import Agent, tool
from strands.models.openai import OpenAIModel

load_dotenv()

app = BedrockAgentCoreApp()

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")


@tool
def extract_clauses(text: str) -> dict:
    """Extract key clauses from contract text."""
    # In production this would use NLP or a specialised model.
    # For this example we return a structured placeholder.
    return {"clauses": ["termination", "liability", "indemnification"]}


model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer is returned.
    params={"extra_body": {"include_reasoning": False}},
)
agent = Agent(
    model=model,
    tools=[extract_clauses],
    system_prompt="You are a document analysis assistant. Extract and summarize key contract clauses.",
)


@app.entrypoint
def handle(payload):
    response = agent(payload.get("prompt"))
    return response.message["content"][0]["text"]


if __name__ == "__main__":
    app.run()

The packages the deployed container needs:

In [ ]:
%%writefile requirements-groq.txt
# Packages installed INSIDE the deployed AgentCore container (Groq edition)
bedrock-agentcore>=1.24.0,<2.0.0
strands-agents[openai]>=1.60.0,<2.0.0
python-dotenv>=1.0.0

## Step 2: Test locally

In a **separate terminal**, in this folder:

```bash
python agent_groq.py
```

This starts a local server on port 8080 with:
- `POST /invocations` — send requests to your agent
- `GET /ping` — health check

Then:

```bash
curl -X POST http://localhost:8080/invocations \
  -H "Content-Type: application/json" \
  -d '{"prompt": "What are the key clauses in a standard NDA?"}'
```

Stop the server with Ctrl+C when you're happy.

## Step 3: Configure the deployment

`configure` generates a Dockerfile, creates an IAM execution role and an ECR repository.

The toolkit packages the folder it runs in, and reuses a `Dockerfile` it finds there instead of generating one. This folder's `Dockerfile` is the reference copy for `agent.py` (the Bedrock edition), so the Groq agent is configured and launched from its own `build_groq/` folder.

In [ ]:
import os
import shutil

from boto3.session import Session
from bedrock_agentcore_starter_toolkit import Runtime

region = Session().region_name or "us-east-1"
print(f"Deploying to region: {region}")

# Build the Groq agent in its own folder (this folder's Dockerfile runs agent.py)
NOTEBOOK_DIR = os.getcwd()
BUILD_DIR = os.path.join(NOTEBOOK_DIR, "build_groq")
os.makedirs(BUILD_DIR, exist_ok=True)
for f in ("agent_groq.py", "requirements-groq.txt"):
    shutil.copy(f, BUILD_DIR)

runtime = Runtime()
os.chdir(BUILD_DIR)  # the toolkit configures the folder it runs in
try:
    runtime.configure(
        entrypoint="agent_groq.py",
        requirements_file="requirements-groq.txt",
        auto_create_execution_role=True,
        auto_create_ecr=True,
        region=region,
        agent_name="doc_analysis_agent_groq",
    )
finally:
    os.chdir(NOTEBOOK_DIR)

## Step 4: Launch

Builds the image with CodeBuild (ARM64), pushes it to ECR, and creates the runtime. The first launch takes a few minutes.

The deployed agent reads `GROQ_API_KEY` from its environment, so the launch passes it as an environment variable. That is fine for a lab; for production, keep the key in AWS Secrets Manager (or an AgentCore Identity API-key credential provider) and read it at startup, so it never appears in the runtime's configuration.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

# The deployed agent reads GROQ_API_KEY and GROQ_MODEL_ID (see agent_groq.py); pass them through
MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")

os.chdir(BUILD_DIR)  # CodeBuild uploads the folder the toolkit runs in
try:
    launch_result = runtime.launch(
        auto_update_on_conflict=True,
        env_vars={"GROQ_API_KEY": os.environ["GROQ_API_KEY"], "GROQ_MODEL_ID": MODEL_ID},
    )
finally:
    os.chdir(NOTEBOOK_DIR)

print(f"Agent ID:  {launch_result.agent_id}")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"ECR URI:   {launch_result.ecr_uri}")

## Step 5: Wait until the endpoint is READY

In [ ]:
import time

status = runtime.status().endpoint["status"]
print(f"Status: {status}")
while status not in ["READY", "CREATE_FAILED", "UPDATE_FAILED"]:
    time.sleep(10)
    status = runtime.status().endpoint["status"]
    print(f"Status: {status}")

print("\nEndpoint is ready." if status == "READY" else "\nDeployment failed — check the console / CloudWatch logs.")

## Step 6: Invoke with the starter toolkit

In [ ]:
response = runtime.invoke(
    {"prompt": "Analyze this contract clause: The vendor shall be liable for direct damages only, excluding consequential, incidental, or punitive damages."}
)
print(response["response"][0] if response.get("response") else response)

## Step 7: Invoke with boto3

This is how another service (Lambda, ECS, Step Functions) would call your agent in production.

In [ ]:
import json
import uuid

import boto3

client = boto3.client("bedrock-agentcore", region_name=region)

response = client.invoke_agent_runtime(
    agentRuntimeArn=launch_result.agent_arn,
    runtimeSessionId=str(uuid.uuid4()),
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "Summarize the liability and indemnification clauses in this contract."}),
)

session_id = response.get("runtimeSessionId")
print(f"Session ID: {session_id}")

# The body is streamed in chunks
content = [chunk.decode("utf-8") for chunk in response.get("response", [])]
print(f"\nAgent response:\n{json.loads(''.join(content))}")

## Step 8: Stop the session

Each session runs in its own microVM. Stop it when the conversation is over to release compute; otherwise it ends after the idle timeout (15 minutes by default).

In [ ]:
if session_id:
    client.stop_runtime_session(
        agentRuntimeArn=launch_result.agent_arn,
        runtimeSessionId=session_id,
        qualifier="DEFAULT",
    )
    print(f"Session {session_id} stopped.")

## Step 9: Clean up

`destroy` removes the runtime, its endpoint, the CodeBuild project and the execution role. `delete_ecr_repo=True` also deletes the image repository. The `build_groq/` folder is removed last.

In [ ]:
result = runtime.destroy(delete_ecr_repo=True)
print("Removed:", *result.resources_removed, sep="\n  - ")
if result.errors:
    print("Errors:", result.errors)
else:
    shutil.rmtree(BUILD_DIR)  # the build folder from Step 3

---
## What you just did

You took a standard Strands agent, added four lines of AgentCore boilerplate, and deployed it to a managed runtime with session isolation, health checks and scaling — without writing a Dockerfile, a SAM template or a load-balancer configuration.

**Try next:** add a tool to `agent_groq.py` and run Steps 3–5 again to update the deployment.